# Prédiction de notes d'hôtels — Fine-tuning RoBERTa (Google Colab)

**Modèle :** `roberta-base` (125M paramètres) — fine-tuning complet sur les avis d'hôtels.

**Avant d'exécuter :**
1. `Runtime → Change runtime type → GPU (T4)`
2. Uploader les 3 fichiers CSV via la cellule Section 2

**Pipeline :**
1. Installation & imports
2. Upload des données
3. Tokenisation
4. Dataset PyTorch
5. Modèle + entraînement
6. Évaluation & comparaison

## 0. Installation

Installe les bibliothèques nécessaires sur Colab.

In [ ]:
# Colab a déjà torch et sklearn — on installe uniquement transformers si absent
!pip install transformers -q

## 1. Imports & détection GPU

Sur Colab avec GPU activé, `device` sera `cuda` — l'entraînement sera ~10x plus rapide que sur CPU.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os, time
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Sur Colab GPU : cuda | Sur Colab CPU : cpu
# MPS (Mac) retiré — non disponible sur Colab
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go')
print(f'PyTorch: {torch.__version__}')

## 2. Upload des données

Deux options pour charger les CSV sur Colab :

**Option A (recommandée) — Upload manuel :** exécuter la cellule ci-dessous,
cliquer sur 'Choisir des fichiers' et uploader les 3 CSV.

**Option B — Google Drive :** monter Drive et lire les fichiers depuis votre Drive.

In [ ]:
# ── Option A : Upload direct (recommandé) ────────────────────────────────────
from google.colab import files

print('Uploader les 3 fichiers : train_hotel_reviews.csv, valid_hotel_reviews.csv, test_hotel_reviews.csv')
uploaded = files.upload()  # Une boîte de dialogue s'ouvre
print('Fichiers uploadés :', list(uploaded.keys()))

In [ ]:
# ── Option B : Depuis Google Drive (décommenter si vous préférez Drive) ──────
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_PATH = '/content/drive/MyDrive/hotel_reviews/'  # adapter le chemin

# ── Chargement des CSV ───────────────────────────────────────────────────────
# Les fichiers uploadés sont dans /content/ sur Colab
train_df = pd.read_csv('train_hotel_reviews.csv')
valid_df = pd.read_csv('valid_hotel_reviews.csv')
test_df  = pd.read_csv('test_hotel_reviews.csv')

# Labels : convertir notes 1-5 → indices 0-4 (obligatoire pour PyTorch CrossEntropyLoss)
train_df['label'] = train_df['Rating'] - 1
valid_df['label'] = valid_df['Rating'] - 1
test_df['label']  = test_df['Rating']  - 1

print(f'Train : {len(train_df)} | Valid : {len(valid_df)} | Test : {len(test_df)}')
print(f'Distribution labels (train) :')
print(train_df['label'].value_counts().sort_index())

## 3. Tokenisation

Le tokenizer RoBERTa découpe chaque texte en tokens et produit :
- `input_ids` : indices des tokens dans le vocabulaire (50 265 mots)
- `attention_mask` : 1 = vrai token, 0 = padding

`max_length=256` : on tronque à 256 tokens. Suffisant pour la majorité des avis,
divise la mémoire GPU par 2 par rapport à 512.

In [ ]:
MODEL_NAME = 'roberta-base'
MAX_LEN    = 256

print('Chargement du tokenizer...')
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    """Convertit une liste de textes en tenseurs input_ids + attention_mask."""
    return tokenizer(
        texts,
        max_length=MAX_LEN,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )

print('Tokenisation en cours (peut prendre 1-2 min)...')
train_enc = tokenize(train_df['Review'].astype(str).tolist())
valid_enc = tokenize(valid_df['Review'].astype(str).tolist())
test_enc  = tokenize(test_df['Review'].astype(str).tolist())

print(f'Shape train : {train_enc["input_ids"].shape}  ← (nb_exemples × max_length)')
print('Tokenisation terminée.')

## 4. Dataset et DataLoader

- `batch_size=32` : sur GPU T4 de Colab, on peut utiliser 32 (vs 16 sur CPU).
  Augmenter à 64 si la VRAM le permet, réduire à 16 si out-of-memory.
- `shuffle=True` sur train : mélanger les données à chaque époque améliore la généralisation.

In [ ]:
class HotelDataset(Dataset):
    """Encapsule les données tokenisées pour PyTorch."""
    def __init__(self, encodings, labels):
        self.input_ids      = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids'      : self.input_ids[idx],
            'attention_mask' : self.attention_mask[idx],
            'labels'         : self.labels[idx]
        }

BATCH_SIZE = 32  # GPU Colab T4 : 32 est optimal. Réduire à 16 si out-of-memory.

train_loader = DataLoader(HotelDataset(train_enc, train_df['label'].tolist()), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(HotelDataset(valid_enc, valid_df['label'].tolist()), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(HotelDataset(test_enc,  test_df['label'].tolist()),  batch_size=BATCH_SIZE, shuffle=False)

print(f'Batches train : {len(train_loader)} | valid : {len(valid_loader)} | test : {len(test_loader)}')

## 5. Chargement du modèle RoBERTa

`RobertaForSequenceClassification` = RoBERTa + une couche linéaire (768 → 5 classes)
ajoutée et initialisée aléatoirement. C'est cette couche + les poids de RoBERTa
qui seront fine-tunés ensemble.

Les warnings UNEXPECTED/MISSING sont normaux : ils indiquent que la tête de
classification est nouvelle (non présente dans le checkpoint pré-entraîné).

In [ ]:
print('Chargement de roberta-base...')
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5  # 5 classes (labels 0 à 4)
)
model = model.to(device)

nb_params = sum(p.numel() for p in model.parameters())
print(f'Modèle sur : {device}')
print(f'Paramètres : {nb_params:,} (~{nb_params/1e6:.0f}M)')

## 6. Optimiseur et scheduler

- **AdamW** avec `lr=2e-5` : learning rate standard pour le fine-tuning de transformers.
  Trop grand → le modèle oublie ses connaissances (catastrophic forgetting).
- **Warmup linéaire** : le LR monte doucement sur les 10% premiers steps,
  puis redescend. Stabilise les premières étapes d'entraînement.

In [ ]:
EPOCHS = 3
LR     = 2e-5

optimizer     = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * 0.1)
scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f'Époques : {EPOCHS} | LR : {LR} | Steps : {total_steps} | Warmup : {warmup_steps}')

## 7. Entraînement

À chaque époque :
1. **Train** : forward → loss → backward → mise à jour des poids
2. **Valid** : évaluation sans gradient (`no_grad`)

Le meilleur modèle selon le F1 macro valid est sauvegardé dans `best_roberta.pt`.

> Sur GPU T4 Colab : ~5-8 min par époque → ~20 min total pour 3 époques.

In [ ]:
def evaluate(model, loader):
    """Évalue le modèle sur un loader. Retourne prédictions, labels, loss moyenne."""
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(device)
            mask  = batch['attention_mask'].to(device)
            lbls  = batch['labels'].to(device)
            out   = model(input_ids=ids, attention_mask=mask, labels=lbls)
            total_loss += out.loss.item()
            all_preds.extend(out.logits.argmax(-1).cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), total_loss / len(loader)


history  = {'train_loss': [], 'valid_loss': [], 'acc': [], 'f1': []}
best_f1  = 0
SAVE_PATH = 'best_roberta.pt'

print('=' * 60)
print(f'Entraînement sur {device} — {EPOCHS} époques')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)

        optimizer.zero_grad()
        out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Évite les gradients explosifs
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(f'  Ep {epoch} | Step {step+1}/{len(train_loader)} | Loss {loss.item():.4f} | {time.time()-t0:.0f}s')

    preds_v, labels_v, valid_loss = evaluate(model, valid_loader)
    acc = accuracy_score(labels_v, preds_v)
    f1  = f1_score(labels_v, preds_v, average='macro')

    history['train_loss'].append(train_loss / len(train_loader))
    history['valid_loss'].append(valid_loss)
    history['acc'].append(acc)
    history['f1'].append(f1)

    print(f'\nÉpoque {epoch}/{EPOCHS} — Train loss: {train_loss/len(train_loader):.4f} | '
          f'Valid loss: {valid_loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | {time.time()-t0:.0f}s')

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  → Meilleur modèle sauvegardé (F1={best_f1:.4f})')

    print('-' * 60)

## 8. Courbes d'apprentissage

Si la valid loss remonte alors que la train loss baisse → overfitting.
3 époques est généralement suffisant pour éviter ça avec RoBERTa.

In [ ]:
ep = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ep, history['train_loss'], 'o-', label='Train', color='steelblue')
axes[0].plot(ep, history['valid_loss'], 'o-', label='Valid',  color='coral')
axes[0].set_title('Loss par époque') ; axes[0].set_xlabel('Époque') ; axes[0].legend()

axes[1].plot(ep, history['f1'],  'o-', label='F1 macro', color='seagreen')
axes[1].plot(ep, history['acc'], 'o-', label='Accuracy', color='purple')
axes[1].set_title('Métriques valid') ; axes[1].set_xlabel('Époque') ; axes[1].legend()

plt.tight_layout() ; plt.show()
print(f'Meilleur F1 : {best_f1:.4f}')

## 9. Évaluation finale — Meilleur modèle

Chargement du meilleur checkpoint et évaluation sur valid + test.

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))

preds_v, labels_v, _ = evaluate(model, valid_loader)
preds_t, labels_t, _ = evaluate(model, test_loader)

acc_v = accuracy_score(labels_v, preds_v) ; f1_v = f1_score(labels_v, preds_v, average='macro')
acc_t = accuracy_score(labels_t, preds_t) ; f1_t = f1_score(labels_t, preds_t, average='macro')

print('=== VALIDATION ===')
print(f'Accuracy : {acc_v:.4f} ({acc_v*100:.2f}%) | F1 macro : {f1_v:.4f}')
print(classification_report(labels_v, preds_v, target_names=['1★','2★','3★','4★','5★']))

print('=== TEST ===')
print(f'Accuracy : {acc_t:.4f} ({acc_t*100:.2f}%) | F1 macro : {f1_t:.4f}')
print(classification_report(labels_t, preds_t, target_names=['1★','2★','3★','4★','5★']))

## 10. Matrice de confusion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, p, l, title, cmap in [
    (axes[0], preds_v, labels_v, 'Validation', 'Blues'),
    (axes[1], preds_t, labels_t, 'Test',       'Greens')
]:
    ConfusionMatrixDisplay(confusion_matrix(l, p), display_labels=['1★','2★','3★','4★','5★']).plot(cmap=cmap, ax=ax)
    ax.set_title(f'Confusion — {title}')
plt.tight_layout() ; plt.show()

## 11. Comparaison — Toutes les approches

In [ ]:
results = pd.DataFrame({
    'Modèle'         : ['TF-IDF + LogReg', 'TF-IDF + LinearSVC', 'SentenceTransformers + LogReg', 'RoBERTa fine-tuné'],
    'Acc valid (%)'  : [63.9,  60.4,  63.1,  round(acc_v*100, 1)],
    'F1 macro valid' : [0.516, 0.489, 0.510, round(f1_v, 3)],
    'F1 macro test'  : [0.493, '—',   0.566, round(f1_t, 3)],
})
print(results.to_string(index=False))

# Graphique
labels_plot = ['TF-IDF\n+LogReg', 'TF-IDF\n+SVC', 'SentTrans\n+LogReg', 'RoBERTa\nfine-tuné']
f1v = [0.516, 0.489, 0.510, f1_v]
f1t = [0.493, 0,     0.566, f1_t]
x   = np.arange(len(labels_plot)) ; w = 0.3

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, f1v, w, label='F1 valid', color=['steelblue','steelblue','seagreen','darkorange'])
b2 = ax.bar(x + w/2, f1t, w, label='F1 test',  color=['#7fb3d3','#7fb3d3','#76c893','#f4a261'])
ax.set_xticks(x) ; ax.set_xticklabels(labels_plot)
ax.set_ylim(0, 1) ; ax.set_ylabel('F1 macro') ; ax.legend()
ax.set_title('Comparaison F1 macro — Toutes approches')
for bar, val in zip(list(b1)+list(b2), f1v+f1t):
    if val > 0:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.3f}', ha='center', fontsize=9)
plt.tight_layout() ; plt.show()

## 12. Analyse des erreurs

Examen des cas mal classés sur le jeu de validation.

In [ ]:
valid_df2 = valid_df.copy()
valid_df2['pred'] = preds_v + 1  # Reconvertir 0-4 → 1-5
errors = valid_df2[valid_df2['Rating'] != valid_df2['pred']].copy()
errors['ecart'] = (errors['Rating'] - errors['pred']).abs()

print(f'Erreurs : {len(errors)} / {len(valid_df2)} ({len(errors)/len(valid_df2)*100:.1f}%)')
print('Répartition par écart :')
print(errors['ecart'].value_counts().sort_index())

errors['ecart'].value_counts().sort_index().plot(
    kind='bar', color='darkorange', edgecolor='black', figsize=(6,3)
)
plt.title('Écart note réelle vs prédite (erreurs)') ; plt.xlabel('Écart') ; plt.tight_layout() ; plt.show()

print('\n=== 5 exemples mal classés ===')
for _, row in errors.head(5).iterrows():
    print(f'\nRéel {row["Rating"]}★ → Prédit {row["pred"]}★ (écart {row["ecart"]})')
    print(str(row['Review'])[:250] + '...')
    print('-' * 70)

## 13. Sauvegarde des prédictions et téléchargement

Les prédictions sont sauvegardées en CSV et téléchargeables depuis Colab.

In [ ]:
# Sauvegarde CSV
pd.DataFrame({
    'Review'        : test_df['Review'],
    'Rating_reel'   : labels_t + 1,
    'Rating_predit' : preds_t  + 1,
}).to_csv('predictions_roberta.csv', index=False)

# Télécharger le CSV et le modèle depuis Colab
from google.colab import files
files.download('predictions_roberta.csv')
files.download('best_roberta.pt')  # Optionnel : télécharger le modèle entraîné

print('=== RÉSUMÉ FINAL ===')
print(f'Modèle           : roberta-base fine-tuné ({EPOCHS} époques)')
print(f'Valid Accuracy   : {acc_v:.4f} ({acc_v*100:.2f}%)')
print(f'Valid F1 macro   : {f1_v:.4f}')
print(f'Test  Accuracy   : {acc_t:.4f} ({acc_t*100:.2f}%)')
print(f'Test  F1 macro   : {f1_t:.4f}')
print(f'Gain vs TF-IDF   : F1 {f1_t - 0.493:+.3f}')
print(f'Gain vs SentTrans: F1 {f1_t - 0.566:+.3f}')